# Lesson 2: File Reader

Add the first tool — `read_file` — so the agent can read code files from disk.

This introduces the **tool-calling loop**: when the LLM decides it needs to read a file, it emits a tool call, the SDK executes our function, and feeds the result back to the LLM.

In [ ]:
%pip install openai-agents mermaid-py --upgrade --quiet

<div style='margin-top: 20px; margin-bottom: 20px; text-align: center; display: flex; flex-direction: column; align-items: center;'>
<img src='./images/lesson_4_command_runner.png' width='600' /> 
</div>

In [4]:
import os
from getpass import getpass

from agents import Agent, Runner, function_tool

MODEL = "gpt-5.4-mini"

In [5]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Defining the `read_file` Tool

We use the `@function_tool` decorator to turn a plain Python function into a tool.
The SDK automatically extracts the parameter schema from type hints and the docstring.

In [6]:
@function_tool
def read_file(path: str) -> str:
    """Read the contents of a file at the given path."""
    try:
        with open(path, "r") as f:
            content = f.read()
        if len(content) > 10000:
            content = content[:10000] + "\n... (truncated)"
        return content
    except FileNotFoundError:
        return f"Error: File not found: {path}"
    except Exception as e:
        return f"Error reading file: {e}"

In [7]:
# Create a test file for the agent to read
with open("example.py", "w") as f:
    f.write('''def fibonacci(n):
    """Return the nth Fibonacci number."""
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


if __name__ == "__main__":
    for i in range(10):
        print(f"fib({i}) = {fibonacci(i)}")
''')

In [8]:
agent = Agent(
    name="File Reader",
    instructions="You are a coding assistant that can read files. Use the read_file tool when asked about file contents.",
    model=MODEL,
    tools=[read_file],
)

In [9]:
result = await Runner.run(agent, "Read example.py and explain what it does.")
print(result.final_output)

`example.py` defines a recursive Fibonacci function and then prints the first 10 Fibonacci numbers.

### What it does

- `fibonacci(n)`:
  - Returns `n` when `n` is `0` or `1`.
  - Otherwise, it computes:
    - `fibonacci(n - 1) + fibonacci(n - 2)`

This matches the standard Fibonacci sequence:

- `0, 1, 1, 2, 3, 5, 8, ...`

### Main block

When the file is run directly:

```python
if __name__ == "__main__":
```

it loops from `0` to `9` and prints each Fibonacci number:

```python
fib(0) = 0
fib(1) = 1
fib(2) = 1
fib(3) = 2
...
fib(9) = 34
```

### Notes

- This implementation is simple but inefficient for larger `n` because it recalculates the same values many times.
- It also uses recursion, so very large inputs may be slow or hit recursion limits.


In [10]:
# Clean up
os.remove("example.py")

## Summary

- Defined a `read_file` tool using `@function_tool` — the SDK auto-generates the JSON schema from type hints.
- The agent decides *when* to call the tool based on the user's request.
- The tool-calling loop: LLM emits tool call → SDK runs our function → result fed back to LLM.
- Next lesson: add `list_files` so the agent can explore directories before reading.